# LightGBM LambdaRank Reranker (GPU + Optuna + F0.5 Calibration + ROC Curves)
Optimized for Precision-heavy Metric (F0.5) with Singleton Handling.

This notebook preserves the query/group logic (no random row mixing).


### 1. Imports and Configuration
Loading necessary libraries and setting static parameters (GPU, Number of Trials, Validation split).


In [20]:
import os, gc, time, json, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve

import warnings

# Suppress all future and runtime warnings globally
warnings.filterwarnings('ignore')


try:
    import optuna
except ImportError:
    os.system("pip install -q optuna")
    import optuna

try:
    from tqdm.auto import tqdm
except ImportError:
    os.system("pip install -q tqdm")
    from tqdm.auto import tqdm

import joblib

DATA_PATH = Path('/kaggle/input/datasets/prince3022272/training-data/training_features_full.csv')
OUTPUT_DIR = Path('/kaggle/working/outputs/lgbm_lambdarank')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
VALIDATION_FRACTION = 0.2
N_TRIALS = 20
USE_GPU = True


### 2. Loading Data
Loads the processed pairwise features directly generated from ANN extraction.


In [21]:
print("[INFO] Loading data in chunks to save RAM...")
chunk_size = 1000000  # 1 million rows at a time
chunks = []

for chunk in pd.read_csv(DATA_PATH, dtype={"query_id": str, "candidate_id": str}, chunksize=chunk_size):
    chunk['label'] = chunk['label'].astype(np.int8)
    feature_cols = [c for c in chunk.columns if c not in ['query_id', 'candidate_id', 'label']]
    for c in feature_cols:
        chunk[c] = pd.to_numeric(chunk[c], errors='coerce').fillna(0.0).astype(np.float32)
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()

FEATURE_COLUMNS = feature_cols
print(f"Shape: {df.shape} | Features: {len(FEATURE_COLUMNS)}")
print(f"Total RAM usage of dataframe: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")


[INFO] Loading data in chunks to save RAM...
Shape: (22068210, 36) | Features: 33
Total RAM usage of dataframe: 5.62 GB


### 3. Query Partitioned Split
Very Important: We split by `query_id` ensuring a Source-1 and all its candidates go exactly into train OR val, never mixed!


In [22]:
def query_partitioned_split(dataset_df, val_fraction=0.2, random_state=42):
    unique_queries = np.array(sorted(dataset_df["query_id"].unique()))
    rng = np.random.default_rng(random_state)
    rng.shuffle(unique_queries)
    val_size = max(1, int(len(unique_queries) * val_fraction))
    val_query_set = set(unique_queries[:val_size])
    
    train_df = dataset_df[~dataset_df["query_id"].isin(val_query_set)].copy()
    val_df = dataset_df[dataset_df["query_id"].isin(val_query_set)].copy()
    
    train_df = train_df.sort_values("query_id").reset_index(drop=True)
    val_df = val_df.sort_values("query_id").reset_index(drop=True)
    return train_df, val_df

train_df, val_df = query_partitioned_split(df, val_fraction=VALIDATION_FRACTION, random_state=RANDOM_STATE)
X_train, y_train = train_df[FEATURE_COLUMNS], train_df["label"].to_numpy(dtype=np.int32)
X_val, y_val = val_df[FEATURE_COLUMNS], val_df["label"].to_numpy(dtype=np.int32)


### 4. Custom Metrics & Threshold Tuner
Here we define exactly how F0.5 is calculated, treating singletons rigorously. We also do exhaustive threshold sweep to find the absolute best F0.5 cutoff point, never assuming 0.5!


In [23]:
# Pre-compute Ground Truth Dicts for Evaluation
val_y_true_by_query = val_df[val_df['label'] == 1].groupby('query_id')['candidate_id'].apply(set).to_dict()
val_all_queries = set(val_df['query_id'].unique())

def macro_f05(y_true_by_query, pred_by_query, all_queries):
    scores = []
    for qid in all_queries:
        true_set = y_true_by_query.get(qid, set())
        pred_set = set(pred_by_query.get(qid, []))
        if len(true_set) == 0:
            score = 1.0 if len(pred_set) == 0 else 0.0
        else:
            tp = len(true_set & pred_set)
            precision = tp / len(pred_set) if pred_set else 0.0
            recall = tp / len(true_set)
            if precision == 0 and recall == 0:
                score = 0.0
            else:
                score = (1.25 * precision * recall) / (0.25 * precision + recall)
        scores.append(score)
    return np.mean(scores)

def exhaustive_threshold_search(val_preds_df, fast_mode=False):
    print("[INFO] Running Exhaustive Threshold Sweep...")
    val_preds_df = val_preds_df.sort_values(["query_id", "score"], ascending=[True, False])
    val_preds_df['rank'] = val_preds_df.groupby('query_id').cumcount() + 1
    val_preds_df['top_score'] = val_preds_df.groupby('query_id')['score'].transform('max')
    
    best_f05 = -1
    best_config = {}
    
    margin_sweep = [0.0, 0.5, 1.0] if fast_mode else np.arange(0.0, 5.0, 0.1)
    for margin in tqdm(margin_sweep, desc="Margin Sweep", disable=fast_mode):
        valid_preds = val_preds_df[(val_preds_df['score'] >= (val_preds_df['top_score'] - margin))]
        pred_dict = valid_preds.groupby("query_id")["candidate_id"].apply(list).to_dict()
        score = macro_f05(val_y_true_by_query, pred_dict, val_all_queries)
        if score > best_f05: 
            best_f05 = score; best_config = {'type': 'margin', 'margin': margin}
            
    unique_scores = np.sort(val_preds_df['score'].unique())[::-1]
    pct_count = 20 if fast_mode else 100
    test_thresh = np.percentile(unique_scores, np.linspace(0, 100, pct_count))
    k_list = [5, 10] if fast_mode else [1, 2, 3, 5, 10, 20]
    
    for t in tqdm(test_thresh, desc="Global Sweep", disable=fast_mode):
        for k in k_list:
            valid_preds = val_preds_df[(val_preds_df['score'] >= t) & (val_preds_df['rank'] <= k)]
            pred_dict = valid_preds.groupby("query_id")["candidate_id"].apply(list).to_dict()
            score = macro_f05(val_y_true_by_query, pred_dict, val_all_queries)
            if score > best_f05:
                best_f05 = score; best_config = {'type': 'global_k', 'threshold': t, 'max_k': k}
                
    return best_f05, best_config


### 5. Hyperparameter Tuning (Optuna)
We tune depth, learning rate, leaf count, regularization etc. The objective explicitly evaluates `macro_f05` after applying a fast threshold calibration.


In [26]:
try:
    import lightgbm as lgb
except ImportError:
    os.system("pip install -q lightgbm")
    import lightgbm as lgb

train_groups = train_df.groupby("query_id", sort=False).size().to_numpy(dtype=np.int32)
val_groups = val_df.groupby("query_id", sort=False).size().to_numpy(dtype=np.int32)

print("[INFO] Tuning LightGBM LambdaRank...")
def objective(trial):
    params = {
        "objective": "lambdarank",
        "metric": "ndcg",
        "device_type": "gpu" if USE_GPU else "cpu",
        "lambdarank_truncation_level": trial.suggest_categorical("lambdarank_truncation_level", [10, 20, 50]),
        "ndcg_at": [1, 3, 5, 10, 20],
        "learning_rate": trial.suggest_categorical("learning_rate", [0.02, 0.03, 0.05]),
        "n_estimators": 500,
        "num_leaves": trial.suggest_categorical("num_leaves", [15, 31, 63]),
        "max_depth": trial.suggest_categorical("max_depth", [-1, 6, 10]),
        "min_child_samples": trial.suggest_categorical("min_child_samples", [20, 50, 100]),
        "subsample": trial.suggest_categorical("subsample", [0.7, 0.85, 1.0]),
        "colsample_bytree": trial.suggest_categorical("colsample_bytree", [0.7, 0.85, 1.0]),
        "reg_alpha": trial.suggest_categorical("reg_alpha", [0, 0.1, 1.0]),
        "reg_lambda": trial.suggest_categorical("reg_lambda", [0.1, 1.0, 5.0]),
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        "verbose": -1
    }
    ranker = lgb.LGBMRanker(**params)
    callbacks = [lgb.early_stopping(stopping_rounds=30, verbose=False)]
    try:
        ranker.fit(X_train, y_train, group=train_groups, eval_set=[(X_val, y_val)], eval_group=[val_groups], callbacks=callbacks)
    except Exception as e:
        print(f"[WARNING] Trial pruned due to LightGBM C++ Error: {e}")
        raise optuna.TrialPruned()

    
    val_preds = val_df[['query_id', 'candidate_id', 'label']].copy()
    val_preds['score'] = ranker.predict(X_val)
    best_f05, _ = exhaustive_threshold_search(val_preds, fast_mode=True)
    return best_f05

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f"\n🏆 Best Macro F0.5: {study.best_value:.5f}")
print(f"🏆 Best Params: {study.best_params}")


[I 2026-09-29 01:16:01,416] A new study created in memory with name: no-name-68afda15-405f-4e65-93e0-4798ce063081


[INFO] Tuning LightGBM LambdaRank...


  0%|          | 0/20 [00:00<?, ?it/s]

[INFO] Running Exhaustive Threshold Sweep...
[I 2026-09-29 01:32:44,947] Trial 0 finished with value: 0.9306037779317811 and parameters: {'lambdarank_truncation_level': 10, 'learning_rate': 0.02, 'num_leaves': 15, 'max_depth': 10, 'min_child_samples': 50, 'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_alpha': 1.0, 'reg_lambda': 5.0}. Best is trial 0 with value: 0.9306037779317811.
[INFO] Running Exhaustive Threshold Sweep...
[I 2026-09-29 01:49:28,283] Trial 1 finished with value: 0.9312312614972221 and parameters: {'lambdarank_truncation_level': 50, 'learning_rate': 0.02, 'num_leaves': 15, 'max_depth': 10, 'min_child_samples': 50, 'subsample': 0.7, 'colsample_bytree': 0.85, 'reg_alpha': 1.0, 'reg_lambda': 0.1}. Best is trial 1 with value: 0.9312312614972221.
[INFO] Running Exhaustive Threshold Sweep...
[I 2026-09-29 02:07:16,729] Trial 2 finished with value: 0.939711868661389 and parameters: {'lambdarank_truncation_level': 20, 'learning_rate': 0.02, 'num_leaves': 31, 'max_depth': 6, 

### 6. Retrain Best Model & Extract Matches
Using the best parameters found by Optuna, we train for longer iterations, explicitly save the model `.txt`/`.cbm` weights, do a deep threshold sweep, and save all artifacts (`model.joblib`, `predictions.csv`).


In [ ]:
# === RETRAIN BEST MODEL ===
best_params = study.best_params
best_params.update({"objective": "lambdarank", "metric": "ndcg", "device_type": "gpu" if USE_GPU else "cpu", "ndcg_at": [1, 3, 5, 10, 20], "n_estimators": 1000, "n_jobs": -1, "random_state": RANDOM_STATE})

print("[INFO] Retraining final model with best params on GPU...")
final_ranker = lgb.LGBMRanker(**best_params)
final_ranker.fit(X_train, y_train, group=train_groups, eval_set=[(X_val, y_val)], eval_group=[val_groups], callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)])

weights_path = OUTPUT_DIR / "lgbm_lambdarank_weights.txt"
final_ranker.booster_.save_model(str(weights_path))
print(f"\n[SUCCESS] Model Weights successfully saved to: {weights_path}\n")

val_preds = val_df[['query_id', 'candidate_id', 'label']].copy()
val_preds['score'] = final_ranker.predict(X_val)
final_f05, final_thresh_config = exhaustive_threshold_search(val_preds, fast_mode=False)

print("\n" + "="*60)
print("FINAL OPTIMIZED PIPELINE")
print("="*60)
print(f"Macro F0.5    : {final_f05:.6f}")
print(f"Threshold Strategy : {final_thresh_config}")

val_preds.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)
model_path = OUTPUT_DIR / "model.joblib"
joblib.dump({"model": final_ranker, "features": FEATURE_COLUMNS, "threshold": final_thresh_config, "val_macro_f05": final_f05}, model_path)
print(f"[SUCCESS] Full Pipeline Saved to {model_path}")


### 7. Evaluation Visuals
Plots standard ROC-AUC and Precision-Recall Curves based on our validated scores vs ground truth.


In [ ]:
print("[INFO] Generating ROC and Precision-Recall Curves...")
y_true = val_preds['label'].values
y_scores = val_preds['score'].values

fpr, tpr, roc_thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)
precision, recall, pr_thresholds = precision_recall_curve(y_true, y_scores)
pr_auc = auc(recall, precision)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(fpr, tpr, color='darkorange', lw=2.5, label=f'ROC curve (AUC = {roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax1.set_xlim([-0.05, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('Receiver Operating Characteristic (ROC)', fontsize=14, fontweight='bold')
ax1.legend(loc="lower right", fontsize=12)
ax1.grid(True, alpha=0.3)

ax2.plot(recall, precision, color='blue', lw=2.5, label=f'PR curve (AUC = {pr_auc:.4f})')
ax2.set_xlim([-0.05, 1.05])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('Precision-Recall Curve (Crucial for F0.5)', fontsize=14, fontweight='bold')
ax2.legend(loc="lower left", fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_pr_curves.png")
plt.show()

print(f"\n[SUMMARY] Area Under ROC Curve (AUC) : {roc_auc:.4f}")
print(f"[SUMMARY] Area Under PR Curve (AUC)  : {pr_auc:.4f}")
